In [2]:
import pandas as pd
from sqlalchemy import create_engine

# Load the dataset
engine = create_engine("postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/attrition")

# Query the data from the database
query1 = """
SELECT * FROM employee_attrition;
"""
attrition_raw_data = pd.read_sql(query1, engine)
print(attrition_raw_data.head())

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOptionLevel  \
0  ...

In [3]:
import numpy as np

In [4]:
# Convert attrition column to 1/0
attrition_raw_data['Attrition_Flag'] = np.where(attrition_raw_data['Attrition'] == 'Yes', 1, 0)

# Correlation with Attrition
numeric_cols = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'YearsInCurrentRole',
                'YearsSinceLastPromotion', 'TotalWorkingYears', 'JobSatisfaction',
                'EnvironmentSatisfaction', 'WorkLifeBalance', 'DistanceFromHome',
                'NumCompaniesWorked', 'Attrition_Flag']
corelation = attrition_raw_data[numeric_cols].corr()['Attrition_Flag'].sort_values(ascending=False)
print(corelation)

Attrition_Flag             1.000000
DistanceFromHome           0.077924
NumCompaniesWorked         0.043494
YearsSinceLastPromotion   -0.033019
WorkLifeBalance           -0.063939
EnvironmentSatisfaction   -0.103369
JobSatisfaction           -0.103481
YearsAtCompany            -0.134392
Age                       -0.159205
MonthlyIncome             -0.159840
YearsInCurrentRole        -0.160545
TotalWorkingYears         -0.171063
Name: Attrition_Flag, dtype: float64


In [5]:
income_analysis = attrition_raw_data.groupby('Attrition')['MonthlyIncome'].agg(['mean', 'median', 'count']).round(2)
print(income_analysis)

attrition_raw_data['Age_Group'] = pd.cut(attrition_raw_data['Age'], 
                                         bins=[17, 25, 35, 45, 55, 100],
                                         labels=['18-25', '26-35', '36-45', '46-55', '55+'])

overtime_age = (attrition_raw_data.groupby(['OverTime', 'Age_Group'])['Attrition_Flag']
                  .mean()
                  .round(4) * 100)
print(overtime_age)

              mean  median  count
Attrition                        
No         6832.74  5204.0   1233
Yes        4787.09  3202.0    237
OverTime  Age_Group
No        18-25        22.62
          26-35        12.08
          36-45         6.01
          46-55         8.12
          55+          13.33
Yes       18-25        64.10
          26-35        38.99
          36-45        17.04
          46-55        19.70
          55+          23.53
Name: Attrition_Flag, dtype: float64


In [11]:
attrition_raw_data['Attrition_Flag'] = attrition_raw_data['Attrition'].map({'Yes': 1, 'No': 0})

attrition_raw_data['Age_Group'] = pd.cut(attrition_raw_data['Age'], 
                         bins=[17, 25, 35, 45, 55, 100],
                         labels=['18-25', '26-35', '36-45', '46-55', '55+'])

attrition_raw_data['Salary_Group'] = pd.cut(attrition_raw_data['MonthlyIncome'],
                            bins=[0, 3000, 5000, 8000, 12000, float('inf')],
                            labels=['Below 3k', '3k-5k', '5k-8k', '8k-12k', 'Above 12k'])

attrition_raw_data['Tenure_Group'] = pd.cut(attrition_raw_data['YearsAtCompany'],
                            bins=[-1, 2, 5, 10, 20, 100],
                            labels=['0-2 years', '3-5 years', '6-10 years', '11-20 years', '20+ years'])

# 1. Overall Attrition
overall = attrition_raw_data['Attrition_Flag'].value_counts(normalize=True).mul(100).round(2).reset_index()
overall.columns = ['Attrition', 'Percentage']
overall.to_csv("overall_attrition.csv", index=False)

# 2. By Department
dept = attrition_raw_data.groupby('Department')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
dept.columns = ['Department', 'Total_Employees', 'Left', 'Attrition_Rate']
dept['Attrition_Rate'] = (dept['Attrition_Rate'] * 100).round(2)
dept.to_csv("attrition_by_department.csv", index=False)

# 3. By OverTime
ot = attrition_raw_data.groupby('OverTime')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
ot.columns = ['OverTime', 'Total_Employees', 'Left', 'Attrition_Rate']
ot['Attrition_Rate'] = (ot['Attrition_Rate'] * 100).round(2)
ot.to_csv("attrition_by_overtime.csv", index=False)

# 4. By Job Role
jobrole = attrition_raw_data.groupby('JobRole')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
jobrole.columns = ['JobRole', 'Total_Employees', 'Left', 'Attrition_Rate']
jobrole['Attrition_Rate'] = (jobrole['Attrition_Rate'] * 100).round(2)
jobrole.to_csv("attrition_by_jobrole.csv", index=False)

# 5. By Age Group
age = attrition_raw_data.groupby('Age_Group')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
age.columns = ['Age_Group', 'Total_Employees', 'Left', 'Attrition_Rate']
age['Attrition_Rate'] = (age['Attrition_Rate'] * 100).round(2)
age.to_csv("attrition_by_agegroup.csv", index=False)

# 6. By Salary Group
salary = attrition_raw_data.groupby('Salary_Group')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
salary.columns = ['Salary_Group', 'Total_Employees', 'Left', 'Attrition_Rate']
salary['Attrition_Rate'] = (salary['Attrition_Rate'] * 100).round(2)
salary.to_csv("attrition_by_salary.csv", index=False)

# 7. By Marital Status
marital = attrition_raw_data.groupby('MaritalStatus')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
marital.columns = ['MaritalStatus', 'Total_Employees', 'Left', 'Attrition_Rate']
marital['Attrition_Rate'] = (marital['Attrition_Rate'] * 100).round(2)
marital.to_csv("attrition_by_maritalstatus.csv", index=False)

# 8. By Gender
gender = attrition_raw_data.groupby('Gender')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
gender.columns = ['Gender', 'Total_Employees', 'Left', 'Attrition_Rate']
gender['Attrition_Rate'] = (gender['Attrition_Rate'] * 100).round(2)
gender.to_csv("attrition_by_gender.csv", index=False)

# 9. By Years at Company
tenure = attrition_raw_data.groupby('Tenure_Group')['Attrition_Flag'].agg(['count', 'sum', 'mean']).reset_index()
tenure.columns = ['Tenure_Group', 'Total_Employees', 'Left', 'Attrition_Rate']
tenure['Attrition_Rate'] = (tenure['Attrition_Rate'] * 100).round(2)
tenure.to_csv("attrition_by_yearsatcompany.csv", index=False)

print("All summary files exported successfully!")

PermissionError: [Errno 13] Permission denied: 'attrition_by_department.csv'

# Key Insights

- Overall Attrition Rate is 16.12%
Out of 1,470 employees, 237 left the company.
- OverTime is a major driver of attrition
Employees working overtime have a 30% attrition rate, compared to only 10% for those who do not.
- Young employees leave the most
The 18–25 age group has the highest attrition rate at 35.77%.
- When young employees also work overtime, the attrition rate jumps to 64.1%.
New joiners are at high risk
- Employees with 0–2 years at the company have a 29.82% attrition rate — significantly higher than longer-tenured staff.
Low salary is strongly linked to leaving
- Employees earning below ₹3,000 have a 28.61% attrition rate.
Those who left had a much lower average monthly income (₹4,787) compared to those who stayed (₹6,833).
- Sales roles have the highest attrition
Sales Representatives show the highest attrition at 39%, followed by Laboratory Technicians and Human Resources.
- Single employees leave more frequently
Single employees have a 25% attrition rate, more than double that of married employees (12%).
- Job satisfaction matters
Employees with low job satisfaction show higher attrition compared to those with high satisfaction.


# Business Recommendations

- Reduce Overtime burden
Review workload distribution, especially for younger and newer employees. Consider hiring additional staff in high-overtime teams.
- Strengthen onboarding and early retention programs
Focus special attention on employees in their first 2 years. Introduce mentorship and regular check-ins.
- Review compensation for lower salary bands
Benchmark salaries for roles paying below ₹5,000 and consider adjustments where attrition is high.
- Targeted retention plans for Sales and young employees
Create role-specific and age-specific retention strategies, especially for Sales Representatives and the 18–25 age group.
- Improve job satisfaction and work-life balance
Conduct regular satisfaction surveys and act on feedback related to workload, recognition, and career growth.
- Monitor high-risk employee groups
Build a simple early-warning system for employees who are young, new, working overtime, and in high-attrition roles.